# Product Review Sentiment Classification — MLP vs TextCNN vs BiLSTM
**ICT-4442 Deep Learning Mini Project — Phase 2 (Interim)**

Team: Pankhuri Kumari (230911084) · Jambhorkar Arya Sachin (230911088) · Sai Prasad Prusty (230911298)

Run this notebook top-to-bottom in Google Colab (Runtime → Change runtime type → GPU is optional, CPU is fine for these model sizes). `USE_REAL_DATA` is `True` by default here — Colab has internet access, so it will pull the real Amazon Polarity dataset from Hugging Face.


In [1]:
!pip -q install datasets

In [2]:
import re, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

USE_REAL_DATA = True     # Colab has internet -> pulls the real Amazon Polarity dataset
N_SAMPLES = 20000        # 10,000 positive + 10,000 negative, per synopsis
MAX_VOCAB = 20000
MAX_LEN = 100
EMBED_DIM = 100
EPOCHS = 5                # bump this up for the Final phase once you have compute budget to spare


## 1. Dataset acquisition
Samples a balanced 20k subset (10k positive + 10k negative) from the Amazon Polarity dataset.

In [3]:
def load_real_amazon_polarity(n_samples=N_SAMPLES, seed=SEED):
    from datasets import load_dataset
    # The old bare "amazon_polarity" repo id was retired on the HF Hub;
    # it now lives under the fancyzhx namespace.
    ds = load_dataset("fancyzhx/amazon_polarity", split="train")
    df = ds.to_pandas()
    df["text"] = df["title"].fillna("") + ". " + df["content"].fillna("")
    pos = df[df.label == 1].sample(n=n_samples // 2, random_state=seed)
    neg = df[df.label == 0].sample(n=n_samples // 2, random_state=seed)
    out = pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out[["text", "label"]]

def load_synthetic_dev_data(n_samples=600, seed=SEED):
    """Fallback smoke-test data only — do NOT report these numbers. Used to verify
    the pipeline runs when there is no internet access to Hugging Face."""
    rng = random.Random(seed)
    pos_templates = [
        "This {item} is absolutely fantastic, I love it and would buy again.",
        "Great {item}, works perfectly and the quality is excellent.",
        "I am very happy with this {item}, highly recommend to everyone.",
        "Amazing {item}! Exceeded my expectations, five stars.",
    ]
    neg_templates = [
        "This {item} is terrible, it broke after one use, do not buy.",
        "Very disappointed with this {item}, poor quality and overpriced.",
        "The {item} stopped working within a week, waste of money.",
        "Awful {item}, not as described and customer service was rude.",
    ]
    items = ["phone case", "blender", "headphones", "backpack", "laptop stand", "coffee maker"]
    rows = []
    for _ in range(n_samples // 2):
        rows.append((rng.choice(pos_templates).format(item=rng.choice(items)), 1))
        rows.append((rng.choice(neg_templates).format(item=rng.choice(items)), 0))
    rng.shuffle(rows)
    return pd.DataFrame(rows, columns=["text", "label"])

df = load_real_amazon_polarity() if USE_REAL_DATA else load_synthetic_dev_data()
print(df.shape)
df.head()


README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  260MB            

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00001-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  258MB            

amazon_polarity/train-00001-of-00004.par(…): downloading bytes:           |  0.00B            

...

amazon_polarity/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

(20000, 2)


,text,label
0,Please stop. Most authors need credentials of ...,0
1,Lucybear Review. I wish they hadn't made such ...,1
2,Slight Improvement Over S-Video Cable. If your...,1
3,Great for any age!!. I recently purchased this...,1
4,"Sent Wrong CD. I was sent the wrong CD, but th...",0


## 2. Preprocessing pipeline
Lowercasing, HTML/URL stripping, punctuation normalisation (negation cues like `not`/`!`/`?` are kept), then a stratified 80/10/10 train/val/test split.

In [4]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9!?',.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")


train=16000 val=2000 test=2000


In [5]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    result = {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
    print(f"{name:14s} | acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}")
    return result

results = []


## 3. Model 1 — MLP on TF-IDF features
**Owner: Pankhuri Kumari (230911084)**

Classical/feed-forward baseline: unigram+bigram TF-IDF vectors into a small dense network.

In [6]:
vectorizer = TfidfVectorizer(max_features=MAX_VOCAB, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_df.text).toarray()
X_val_tfidf = vectorizer.transform(val_df.text).toarray()
X_test_tfidf = vectorizer.transform(test_df.text).toarray()

mlp = models.Sequential([
    layers.Input(shape=(X_train_tfidf.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
mlp.fit(X_train_tfidf, train_df.label.values,
        validation_data=(X_val_tfidf, val_df.label.values),
        epochs=EPOCHS, batch_size=32, verbose=1)

preds = (mlp.predict(X_test_tfidf) > 0.5).astype(int).ravel()
results.append(evaluate("MLP (TF-IDF)", test_df.label.values, preds))


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.8533 - loss: 0.3410 - val_accuracy: 0.8880 - val_loss: 0.2748
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9686 - loss: 0.0946 - val_accuracy: 0.8790 - val_loss: 0.3546
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9925 - loss: 0.0255 - val_accuracy: 0.8770 - val_loss: 0.4779
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9976 - loss: 0.0077 - val_accuracy: 0.8745 - val_loss: 0.6180
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9995 - loss: 0.0027 - val_accuracy: 0.8725 - val_loss: 0.7493
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
MLP (TF-IDF)   | acc=0.8655  prec=0.8703  rec=0.8590  f1=0.8646


## 4. Shared tokenizer for CNN / BiLSTM
TextCNN and BiLSTM both consume learned word embeddings, so they share one Keras `Tokenizer` + padded sequence pipeline (this keeps the input representation identical between the two, isolating the architecture as the only difference).

In [7]:
tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tok.fit_on_texts(train_df.text)
vocab_size = min(MAX_VOCAB, len(tok.word_index) + 1)

X_train_seq = pad_sequences(tok.texts_to_sequences(train_df.text), maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq = pad_sequences(tok.texts_to_sequences(val_df.text), maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq = pad_sequences(tok.texts_to_sequences(test_df.text), maxlen=MAX_LEN, padding="post", truncating="post")
print("vocab_size:", vocab_size)


vocab_size: 20000


## 5. Model 2 — TextCNN
**Owner: Sai Prasad Prusty (230911298)**

Conv1D filters over learned word embeddings capture local n-gram-like sentiment cues (e.g. “not good”, “highly recommend”).

In [8]:
textcnn = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(vocab_size, EMBED_DIM),
    layers.Conv1D(128, 5, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
textcnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
textcnn.fit(X_train_seq, train_df.label.values,
            validation_data=(X_val_seq, val_df.label.values),
            epochs=EPOCHS, batch_size=32, verbose=1)

preds = (textcnn.predict(X_test_seq) > 0.5).astype(int).ravel()
results.append(evaluate("TextCNN", test_df.label.values, preds))


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7750 - loss: 0.4477 - val_accuracy: 0.8690 - val_loss: 0.3071
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9273 - loss: 0.1940 - val_accuracy: 0.8610 - val_loss: 0.3634
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9827 - loss: 0.0612 - val_accuracy: 0.8645 - val_loss: 0.3906
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9976 - loss: 0.0122 - val_accuracy: 0.8445 - val_loss: 0.5714
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9996 - loss: 0.0031 - val_accuracy: 0.8735 - val_loss: 0.5147
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
TextCNN        | acc=0.8485  prec=0.8374  rec=0.8650  f1=0.8510


## 6. Model 3 — BiLSTM
**Owner: Jambhorkar Arya Sachin (230911088)**

Reads the review sequentially in both directions to capture word order and longer-range context.

In [9]:
bilstm = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(vocab_size, EMBED_DIM),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
bilstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bilstm.fit(X_train_seq, train_df.label.values,
           validation_data=(X_val_seq, val_df.label.values),
           epochs=EPOCHS, batch_size=32, verbose=1)

preds = (bilstm.predict(X_test_seq) > 0.5).astype(int).ravel()
results.append(evaluate("BiLSTM", test_df.label.values, preds))


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.8221 - loss: 0.3914 - val_accuracy: 0.8660 - val_loss: 0.3086
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9281 - loss: 0.1909 - val_accuracy: 0.8630 - val_loss: 0.3586
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9635 - loss: 0.1063 - val_accuracy: 0.8550 - val_loss: 0.4856
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9730 - loss: 0.0795 - val_accuracy: 0.8595 - val_loss: 0.5133
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9849 - loss: 0.0488 - val_accuracy: 0.8350 - val_loss: 0.9482
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
BiLSTM         | acc=0.8255  prec=0.8182  rec=0.8370  f1=0.8275


## 7. Preliminary comparison table
Copy this table into Part B, Section 3 of the Interim report (and keep refining it for the Final report's common comparison table).

In [10]:
res_df = pd.DataFrame(results)
res_df.to_csv("preliminary_results.csv", index=False)
res_df


,model,accuracy,precision,recall,f1
0,MLP (TF-IDF),0.8655,0.870314,0.859,0.864620
1,TextCNN,0.8485,0.837367,0.865,0.850959
2,BiLSTM,0.8255,0.818182,0.837,0.827484


## Notes for the team
- All three models train on the **identical** cleaned text, split, and labels — only the input representation and architecture differ, so any accuracy gap is attributable to architecture, not experimental inconsistency (as promised in the synopsis).
- `EPOCHS = 5` is deliberately small — these are **preliminary, unoptimized** numbers for the Interim phase, per the guidelines. Increase epochs, add early stopping / learning-rate scheduling, and try pretrained embeddings (word2vec/GloVe) for the Final phase.
- Push this notebook (and `preliminary_results.csv`) to the shared GitHub repo with **per-member commits** — the guidelines cross-check individual contribution against commit history, not just the final report.
- For the Final phase, extend Section 6 with an error-analysis pass (sample misclassified reviews, look for sarcasm / negation / mixed sentiment).
